# Ibovespa data cleaning and exploratory analysis

Daily `^BVSP` data run from 2000 through 2025. The main analytical window starts in 2008. The notebook preserves the raw input and records validation flags, missing sessions, and return outliers.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ibovespa_cleaning import clean_data, export

START = "2000-01-01"
END = "2026-01-01"  # exclusive end date, preserving the original study window
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
from local_snapshots import raw_ibovespa_path
CACHE = raw_ibovespa_path()
if not CACHE.is_file():
    raise FileNotFoundError(f"Missing daily data: {CACHE}")

raw_frame = pd.read_csv(CACHE, index_col="Date", parse_dates=True)
print("Raw rows:", len(raw_frame))


## Cleaning, calendar, and audit

The notebook exports the audit tables after cleaning. `df_vol` retains the complete price sequence from 2008, including sessions with unavailable volume.

In [ ]:
result = clean_data(raw_frame, START, END)
df = result["full"].copy()
df_vol = result["analysis"].copy()
export(result, OUTPUT_DIR)
print(json.dumps(result["summary"], indent=2, ensure_ascii=False))
print("\nVolume by year:")
display(result["volume_annual"])
print("\nMissing sessions (review the source and calendar):")
display(result["sessions_missing"])
print("\nRejected rows:")
display(result["rejected"])
print("\nReturns with absolute value > 15% (flagged and retained):")
display(result["outliers"])


In [ ]:
print("Missing observations in the analysis dataset:")
display(df_vol.isna().sum().to_frame("missing"))
display(df_vol.head())
display(df_vol.tail())
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(df_vol.index, df_vol.Close)
ax1.set(title="Ibovespa prices retained since 2008", ylabel="Index points", yscale="log")
ax2.plot(df_vol.index, df_vol.Volume)
ax2.set(ylabel="Reported volume", xlabel="Data")
plt.tight_layout()
plt.show()


## Returns and the contiguous sample

Sequential analyses use the most recent uninterrupted run, starting at the first price after the last missing exchange session since 2008. The full audit preserves the gaps.

In [ ]:
gaps = pd.DatetimeIndex(result["sessions_missing"]["Date"])
gaps = gaps[gaps>=pd.Timestamp("2008-01-01")]
if len(gaps):
    df_vol = df_vol.loc[df_vol.index>gaps.max()].copy()
if df_vol.empty:
    raise ValueError("No observed segment follows the last gap.")
df_vol.loc[df_vol.index[0],["simple_return","log_return"]] = np.nan
if df_vol.simple_return.iloc[1:].isna().any():
    raise ValueError("The selected window contains a gap.")
print(f"contiguous segment: {df_vol.index[0].date()} through {df_vol.index[-1].date()} | {len(df_vol)} sessions")
print(df_vol[["Close","simple_return","log_return"]].head())

## Forecast and execution horizons

The exploratory predictability study uses close-to-close returns. The walk-forward backtest enters at the next opening price and exits at the close of its configured holding period. A single-position rule applies across adjacent test windows.

In [ ]:
# 1. Log return = ln(1 + simple return)
diff = (df_vol["log_return"] - np.log1p(df_vol["simple_return"])).abs().max()
print("Maximum difference:", diff)   # deve ser ~1e-16

# 2. The two measures are nearly equal for small returns.
print(df_vol[["simple_return", "log_return"]].describe())

# 3. Os days mais extremos
print(df_vol[["Close", "simple_return", "log_return"]].nsmallest(5, "simple_return"))
print(df_vol[["Close", "simple_return", "log_return"]].nlargest(5, "simple_return"))

## Descriptive statistics

In [ ]:
# Statistics settings
r  = df_vol["log_return"].dropna()       # log-return
rs = df_vol["simple_return"].dropna()   # simple return
YEAR = 252                             # trading sessions per year

pd.options.display.float_format = "{:.6f}".format

In [ ]:
from scipy import stats
def desc_basica(x):
    qs = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    out = {
        "n": len(x),
        # central tendency
        "mean": x.mean(),
        "median": x.median(),
        "mode_aprox (arred. 0.1%)": x.round(3).mode().iloc[0],
        # dispersion
        "var": x.var(),
        "std": x.std(),
        "mad (median)": stats.median_abs_deviation(x),
        "mad_scaled (~std)": stats.median_abs_deviation(x, scale="normal"),
        "deviation_mean_abs": (x - x.mean()).abs().mean(),
        "min": x.min(),
        "max": x.max(),
        "range": x.max() - x.min(),
        "iqr": x.quantile(0.75) - x.quantile(0.25),
    }
    for p in qs:
        out[f"q{round(p*100):02d}"] = x.quantile(p)

    # distribution shape (population estimates, bias=True)
    out["skewness"] = stats.skew(x, bias=True)
    out["kurtosis_excess"] = stats.kurtosis(x, fisher=True, bias=True)
    out["kurtosis_raw"] = out["kurtosis_excess"] + 3          # <- a do PSR/DSR
    # adjusted pandas estimates for comparison
    out["skewness_pandas (adjusted)"] = x.skew()
    out["kurtosis_excess_pandas (adjusted)"] = x.kurt()
    return pd.Series(out)

tab_desc = pd.DataFrame({"log": desc_basica(r), "simple": desc_basica(rs)})
print(tab_desc.to_string())

In [ ]:
mu, sd = r.mean(), r.std()
z = (r - mu) / sd

# Compare observed k-sigma exceedances with the normal expectation.
tails = []
for k in [2, 3, 4, 5]:
    obs = int((z.abs() > k).sum())
    esp = len(r) * 2 * stats.norm.sf(k)
    tails.append({"k_sigma": k, "observed": obs, "expected_normal": round(esp, 2),
                   "ratio_obs/esp": obs / esp if esp > 0 else np.nan})
print(pd.DataFrame(tails).to_string(index=False))

# Normality tests (rejecting H0 indicates evidence against normality)
jb = stats.jarque_bera(r)
dag = stats.normaltest(r)
print("\nJarque-Bera:", jb.statistic, "p =", jb.pvalue)
print("D'Agostino-Pearson:", dag.statistic, "p =", dag.pvalue)

# Tail asymmetry
print("\nratio q95/|q05|:", r.quantile(0.95) / abs(r.quantile(0.05)))
print("ratio q99/|q01|:", r.quantile(0.99) / abs(r.quantile(0.01)))

In [ ]:
daily_risk_free_rate = 0.0   # Use daily CDI when evaluating a nonzero risk-free rate.

# Annualized return and volatility
ret_annual_aritm = rs.mean() * YEAR
vol_annual = rs.std() * np.sqrt(YEAR)

# Equity curve and drawdown
equity = (1 + rs).cumprod()
cagr = equity.iloc[-1] ** (YEAR / len(rs)) - 1
peak = equity.cummax().clip(lower=1.0)
dd = equity / peak - 1
max_dd = dd.min()

# Longest drawdown in sessions below the previous peak
sub = dd < 0
grp = (sub != sub.shift()).cumsum()
max_dd_dur = int(sub.groupby(grp).sum().max())

# Ratios
sharpe = (rs.mean() - daily_risk_free_rate) / rs.std() * np.sqrt(YEAR)
downside = np.sqrt((np.minimum(rs - daily_risk_free_rate, 0) ** 2).mean())
sortino = (rs.mean() - daily_risk_free_rate) / downside * np.sqrt(YEAR)
calmar = cagr / abs(max_dd)

metrics = pd.Series({
    "return_value_annual_aritmetico": ret_annual_aritm,
    "CAGR": cagr,
    "vol_annual": vol_annual,
    "sharpe_annual": sharpe,
    "daily_sharpe (p/ PSR)": rs.mean() / rs.std(),
    "daily_downside_deviation": downside,
    "sortino_annual": sortino,
    "max_drawdown": max_dd,
    "data_max_drawdown": dd.idxmin(),
    "max_dd_duration_days": max_dd_dur,
    "calmar": calmar,
    "return_value_cumulative": equity.iloc[-1] - 1,
})
print(metrics.to_string())

In [ ]:
def var_cvar(x, level):
    q = x.quantile(1 - level)
    return -q, -x[x <= q].mean()

# Historical
for nv in [0.95, 0.99]:
    v, c = var_cvar(rs, nv)
    print(f"Historical {int(nv*100)}%:  VaR = {v:.4%}   CVaR/ES = {c:.4%}")

# Parametric (assumes normality)
for nv in [0.95, 0.99]:
    zn = stats.norm.ppf(1 - nv)
    print(f"Normal    {int(nv*100)}%:  VaR = {-(rs.mean() + rs.std()*zn):.4%}")

# Cornish-Fisher (corrige assimetria e kurtosis)
S = stats.skew(rs, bias=True)
K = stats.kurtosis(rs, fisher=True, bias=True)   # excesso
for nv in [0.95, 0.99]:
    zn = stats.norm.ppf(1 - nv)
    zcf = (zn + (zn**2 - 1)*S/6 + (zn**3 - 3*zn)*K/24 - (2*zn**3 - 5*zn)*(S**2)/36)
    print(f"Cornish-Fisher {int(nv*100)}%: VaR = {-(rs.mean() + rs.std()*zcf):.4%}")

In [ ]:
def max_streak(cond):
    g = (cond != cond.shift()).cumsum()
    return int(cond.groupby(g).sum().max())

gains = rs[rs > 0]
losses = rs[rs < 0]

perfil = pd.Series({
    "days_positivos": len(gains),
    "days_negativos": len(losses),
    "days_zero": int((rs == 0).sum()),
    "Win rate": len(gains) / len(rs),
    "gain_mean": gains.mean(),
    "loss_mean": losses.mean(),
    "payoff (gain_mean/|loss_mean|)": gains.mean() / abs(losses.mean()),
    "profit_factor": gains.sum() / abs(losses.sum()),
    "expectation_por_day": rs.mean(),
    "best_day": rs.max(),
    "data_best_day": rs.idxmax(),
    "worst_day": rs.min(),
    "data_worst_day": rs.idxmin(),
    "longest_run_high": max_streak(rs > 0),
    "longest_run_low": max_streak(rs < 0),
})
print(perfil.to_string())

# Tail contribution after excluding the N best sessions
for n in [5, 10, 20]:
    sem = rs.drop(rs.nlargest(n).index)
    print(f"Excluding the {n} best sessions, cumulative return = {(1+sem).prod()-1:.2%}  "
          f"(original: {(1+rs).prod()-1:.2%})")

In [ ]:
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

lags = 10
tab_acf = pd.DataFrame({
    "acf_return_value": acf(r, nlags=lags)[1:],
    "acf_return_value^2": acf(r**2, nlags=lags)[1:],
    "acf_|return_value|": acf(r.abs(), nlags=lags)[1:],
}, index=range(1, lags + 1))
tab_acf.index.name = "lag"
print(tab_acf.to_string())
print("\napproximate 95% band under white noise: ±", 1.96 / np.sqrt(len(r)))

print("\nLjung-Box (returns):")
print(acorr_ljungbox(r, lags=[5, 10, 20], return_df=True))
print("\nLjung-Box (returns^2):")
print(acorr_ljungbox(r**2, lags=[5, 10, 20], return_df=True))

adf = adfuller(r)
print("\nADF (stationarity): statistics =", adf[0], " p =", adf[1])

arch = het_arch(r, nlags=10)
print("ARCH test for conditional heteroskedasticity: LM =", arch[0], " p =", arch[1])

In [ ]:
# By year
por_year = pd.DataFrame({
    "return_value_annual": (1 + rs).groupby(rs.index.year).prod() - 1,
    "vol_annual": rs.groupby(rs.index.year).std() * np.sqrt(YEAR),
    "sharpe": rs.groupby(rs.index.year).mean() / rs.groupby(rs.index.year).std() * np.sqrt(YEAR),
    "skew": rs.groupby(rs.index.year).apply(lambda x: stats.skew(x, bias=True)),
    "kurt_raw": rs.groupby(rs.index.year).apply(lambda x: stats.kurtosis(x, fisher=False, bias=True)),
    "worst_day": rs.groupby(rs.index.year).min(),
    "best_day": rs.groupby(rs.index.year).max(),
    "rate_hit": rs.groupby(rs.index.year).apply(lambda x: (x > 0).mean()),
})
print(por_year.to_string())

# By weekday
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
por_day = rs.groupby(rs.index.day_name()).agg(["count", "mean", "std", "median"]).reindex(order)
por_day["rate_hit"] = rs.groupby(rs.index.day_name()).apply(lambda x: (x > 0).mean()).reindex(order)
print("\n", por_day.to_string())

# By month (seasonality)
by_month = rs.groupby(rs.index.month).agg(["count", "mean", "std"])
by_month["rate_hit"] = rs.groupby(rs.index.month).apply(lambda x: (x > 0).mean())
print("\n", by_month.to_string())

In [ ]:
# ANOVA for differences in mean return by weekday
groups = [g.values for _, g in rs.groupby(rs.index.dayofweek)]
print(stats.f_oneway(*groups))
print(stats.kruskal(*groups))   # nonparametric version

In [ ]:
vol21  = rs.rolling(21).std()  * np.sqrt(YEAR)
vol63  = rs.rolling(63).std()  * np.sqrt(YEAR)
vol252 = rs.rolling(252).std() * np.sqrt(YEAR)

print(pd.DataFrame({"vol21": vol21, "vol63": vol63, "vol252": vol252}).describe().to_string())

# Volatilidade ponderada exponencialmente (estilo RiskMetrics, lambda = 0.94)
vol_ewma = np.sqrt((rs**2).ewm(alpha=1 - 0.94, adjust=False).mean()) * np.sqrt(YEAR)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 13))

# 1. Histograma vs normal
ax = axes[0, 0]
ax.hist(r, bins=120, density=True, alpha=0.6, color="tab:blue")
xs = np.linspace(r.min(), r.max(), 500)
ax.plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r-", label="Normal")
ax.set_title("Log-return histogram and normal density")
ax.legend()

# 2. Log-scale histogram highlights the tails.
ax = axes[0, 1]
ax.hist(r, bins=120, density=True, alpha=0.6, color="tab:blue")
ax.plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), "r-")
ax.set_yscale("log")
ax.set_ylim(1e-2)
ax.set_title("Log-return histogram on a logarithmic y-axis")

# 3. QQ-plot
stats.probplot(r, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("QQ-plot vs normal")

# 4. Daily returns and volatility clustering
axes[1, 1].plot(r.index, r, lw=0.5)
axes[1, 1].set_title("Daily log returns")

# 5. Rolling volatility
axes[2, 0].plot(vol21, label="21d", lw=0.8)
axes[2, 0].plot(vol63, label="63d", lw=0.8)
axes[2, 0].plot(vol252, label="252d", lw=1.2)
axes[2, 0].set_title("Annualized rolling volatility")
axes[2, 0].legend()

# 6. Drawdown
axes[2, 1].fill_between(dd.index, dd, 0, color="tab:red", alpha=0.5)
axes[2, 1].set_title("Drawdown")

plt.tight_layout()
plt.show()

# ACF of returns and squared returns
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(r, lags=40, ax=ax[0], title="ACF of returns")
plot_acf(r**2, lags=40, ax=ax[1], title="ACF of squared returns")
plt.tight_layout()
plt.show()

In [ ]:
from legacy_backtest import declust, walk_forward

# parameters
DOWNSIDE_THRESHOLD     = -0.02
VOLATILITY_PERCENTILE       = 0.90
WINDOW_VOL    = 21
MIN_HIST      = 504
N_BOOT        = 10000

HORIZON     = 5
MIN_EVENTS   = 30
ALPHA         = 0.05
SEED          = 42

TRAINING_WINDOW = 756
TEST_WINDOW  = 252

BOOT_BLOCK    = 10
COST_BPS     = 10      # Cost per side, applied at entry and exit.


rng = np.random.default_rng(SEED)
round_trip_cost = 2 * COST_BPS / 1e4

close = df_vol["Close"]
ret = close.pct_change(fill_method=None)
vol = ret.rolling(WINDOW_VOL).std() * np.sqrt(YEAR)

# Volatility cutoff from an expanding history lagged by one session.
lim_vol = vol.expanding(min_periods=MIN_HIST).quantile(VOLATILITY_PERCENTILE).shift(1)
sample = lim_vol.notna() & vol.notna()          # Common sample across conditions.
vol_high = vol >= lim_vol
down = ret < DOWNSIDE_THRESHOLD

conds = {
    "All sessions (sample)":                   sample,
    f"down < {DOWNSIDE_THRESHOLD:.0%}":          sample & down,
    f"High volatility (top {1-VOLATILITY_PERCENTILE:.0%})":   sample & vol_high,
    "Downside and high volatility":                  sample & down & vol_high,
}
N_TESTS = (len(conds) - 1) * 2                  # Three conditions and two horizons with Bonferroni correction.

def return_value_forecast(h):
    return close.shift(-h) / close - 1           # Return from the close at t through t+h.

def boot_diff(f, m, chunk=500):
    """Block bootstrap of paired future returns and conditions, preserving
    serial dependence. Return the distribution of E[condition] - E[baseline]."""
    n = len(f); B = BOOT_BLOCK
    nb = int(np.ceil(n / B)); ar = np.arange(B)
    out = []
    for _ in range(int(np.ceil(N_BOOT / chunk))):
        starts = rng.integers(0, n - B + 1, size=(chunk, nb))
        idx = (starts[:, :, None] + ar).reshape(chunk, -1)[:, :n]
        ff, mm = f[idx], m[idx]
        cnt = mm.sum(axis=1)
        cm = np.where(cnt > 0, (ff * mm).sum(axis=1) / np.maximum(cnt, 1), np.nan)
        out.append(cm - ff.mean(axis=1))
    d = np.concatenate(out)[:N_BOOT]
    return d[~np.isnan(d)]


# PART 1: E[R_{t+h} | X_t] over the full sample

def analisar(h):
    f_all = return_value_forecast(h)
    ok = (sample & f_all.notna()).values
    idx_ok = df_vol.index[ok]
    f = f_all.values[ok]
    base = f.mean()
    cols = {}
    for name, m in conds.items():
        mk = m.values[ok]
        pos = np.flatnonzero(mk)
        n = len(pos)
        if n < 2:
            cols[name] = pd.Series({"n_days": n}); continue
        x = f[pos]
        nd = declust(pos, h)
        epis = int((pd.Series(idx_ok[pos]).diff().dt.days > 10).sum() + 1)

        if name.startswith("All sessions"):
            diff, lo, hi, p, status = 0.0, np.nan, np.nan, np.nan, "(baseline)"
        else:
            d = boot_diff(f, mk)
            lo, hi = np.percentile(d, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
            p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
            diff = x.mean() - base
            if len(nd) < MIN_EVENTS:
                status = "INSUFFICIENT (< MIN_EVENTS)"
            elif p < ALPHA / N_TESTS:
                status = "significativo (Bonferroni)"
            elif p < ALPHA:
                status = "significant before correction"
            else:
                status = "statistically inconclusive"

        cols[name] = pd.Series({
            "n_days": n,
            "n_does_not_sobrepostos": len(nd),
            "episodios (gap>10d)": epis,
            f"E[R_t+{h}]": x.mean(),
            "median": np.median(x),
            "deviation": x.std(),
            "E[R] base (sample)": base,
            "Difference vs. baseline": diff,
            "Block-bootstrap CI lower bound": lo,
            "Block-bootstrap CI upper bound": hi,
            "p_boot (bilateral)": p,
            "prob(R>0)": (x > 0).mean(),
            "worst": x.min(),
            "best": x.max(),
            "veredito": status,
        })
    return pd.DataFrame(cols)

pd.options.display.float_format = "{:.5f}".format
for h in sorted({1, HORIZON}):
    print(f"\n{'='*70}\nHORIZONTE = {h} day(s)   |   sample: {sample.sum()} days\n{'='*70}")
    print(analisar(h).to_string())


# PART 2: Walk-forward training, signals, and testing with costs

OUTPUT_LEGACY = Path("results_legacy")/"P1_execution_D1"
OUTPUT_LEGACY.mkdir(parents=True,exist_ok=True)
(OUTPUT_LEGACY/"configuracao.json").write_text(json.dumps({
    "instrument":"^BVSP","tipo_price":"Index benchmark",
    "window_start":str(df_vol.index[0].date()),"window_end":str(df_vol.index[-1].date()),
    "train":TRAINING_WINDOW,"test":TEST_WINDOW,"horizon":HORIZON,"cost_bps_side":COST_BPS,
    "entry":"Open at t+1","output":"Close at t+h"},indent=2),encoding="utf-8")

for name, use_volatility in [("Downside and high volatility", True), (f"Only returns below {DOWNSIDE_THRESHOLD:.0%}", False)]:
    folds, trades = walk_forward(
        df_vol, ret, vol, horizon=HORIZON, training_window=TRAINING_WINDOW,
        test_window=TEST_WINDOW, downside_threshold=DOWNSIDE_THRESHOLD, volatility_percentile=VOLATILITY_PERCENTILE,
        min_events=MIN_EVENTS, round_trip_cost=round_trip_cost, use_volatility=use_volatility,
    )
    label = "down_e_vol" if use_volatility else "down"
    folds.to_csv(OUTPUT_LEGACY/f"windows_{label}.csv",index=False)
    trades.to_csv(OUTPUT_LEGACY/f"trades_{label}.csv",index=False)
    print(f"\n{'='*70}\nWALK-FORWARD ({name}) | h = {HORIZON} | round-trip cost = {round_trip_cost*1e4:.0f} bps\n{'='*70}")
    print(folds.to_string(index=False))
    if len(trades) < 2:
        print("Insufficient trades. The training signals did not meet the minimum event count.")
    else:
        L = trades["net"]
        t = L.mean() / (L.std() / np.sqrt(len(L)))
        print(f"\ntrades: {len(L)} | mean gross return: {trades['raw'].mean():.4%} | "
              f"mean net return: {L.mean():.4%} | hit: {(L>0).mean():.1%} | t(net vs zero): {t:.2f} | "
              f"net sum: {L.sum():.2%}")